In [22]:
from pathlib import Path

artist_folder = Path(r"artist")

work_folder = Path(r"work")

recording_folder = Path(r"recording")

In [23]:
artist_file = Path(r"artist\mbdump\artist")

with artist_file.open("rb") as file:
    sample = file.read(2000)

# print(sample.decode("utf-8", errors="replace"))

In [24]:
work_file = Path(r"work\mbdump\work")

with work_file.open("rb") as file:
    sample = file.read(2000)

# print(sample.decode("utf-8", errors="replace"))

In [25]:
recording_file = Path(r"recording\mbdump\recording")

with recording_file.open("rb") as file:
    sample = file.read(2000)

In [26]:
import json
import pandas as pd

records = []
rows_scanned = 0

with artist_file.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        rows_scanned += 1
        artist = json.loads(line)

        # Keep US people and groups only.
        if artist.get("country") != "US":
            continue
        if artist.get("type") not in ("Person", "Group"):
            continue

        life_span = artist.get("life-span") or {}
        tags = artist.get("tags")

        records.append({
            "artist_mbid": artist.get("id"),
            "artist_name": artist.get("name"),
            "country": artist.get("country"),
            "artist_type": artist.get("type"),
            "disambiguation": artist.get("disambiguation"),
            "begin_date": life_span.get("begin"),
            "end_date": life_span.get("end"),
            "tags": (
                [tag["name"] for tag in tags
                 if isinstance(tag, dict) and tag.get("name")]
                if isinstance(tags, list) else None
            ),
        })

us_artists_df = pd.DataFrame(records, columns=[
    "artist_mbid", "artist_name", "country", "artist_type",
    "disambiguation", "begin_date", "end_date", "tags",
])

begin_year = pd.to_numeric(
    us_artists_df["begin_date"].astype("string").str[:4],
    errors="coerce",
)

us_artists_df = us_artists_df[
    begin_year.notna() & (begin_year > 2010)
]

print(f"Records scanned: {rows_scanned:,}")
print(f"US artists kept: {len(us_artists_df):,}")
display(us_artists_df.head())

Records scanned: 3,035,083
US artists kept: 16,971


,artist_mbid,artist_name,country,artist_type,disambiguation,begin_date,end_date,tags
6221,58f27fe8-0d3e-4768-a615-ef6a6de88a05,Varsity,US,Group,Indie Pop-Rock band from Chicago,2013-01,NaN,"[fuzz, garage pop, indie rock]"
11365,b9bd94a5-1a32-46d6-9da1-c2d96baf3b1c,Sunshine Boys,US,Group,Chicago three-piece rock band,2016,NaN,[]
12298,ff3abcf7-5ea3-4fba-a50e-239dd66673d7,Glimmer,US,Group,Shoegaze/grunge band,2023-02,NaN,"[grunge, shoegaze]"
15966,2acfdc44-efbd-4730-ab2f-2dec4b90701e,NESkimos,US,Group,,2012,NaN,[]
19635,5e44bf5c-f37b-4b44-8c7f-1fda8eb1d366,Mid-Life Crisis,US,Group,,2015-03-14,NaN,[filk]


In [27]:

import json
import pandas as pd

records = []

with work_file.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            records.append(json.loads(line))

        if len(records) >= 50000:
            break

works_df = pd.json_normalize(records)

print("Sample shape:", works_df.shape)
display(works_df.head())

Sample shape: (50000, 16)


,relations,id,iswcs,language,tags,disambiguation,attributes,languages,aliases,title,annotation,type,genres,type-id,rating.votes-count,rating.value
0,"[{'begin': None, 'target-type': 'artist', 'end...",e0c19b8f-8e9e-3249-8b6c-6b8525a96e95,[],NaN,[],,[],[],[],Paris Takes the Helm,NaN,NaN,[],NaN,0,NaN
1,"[{'begin': None, 'target-type': 'artist', 'end...",cfc88d6b-6306-340f-9ecb-38f40421e81f,[],NaN,[],"Land of the Giants, season 1, main title",[],[],[],Suite From the Crash,NaN,NaN,[],NaN,0,NaN
2,[{'artist': {'disambiguation': 'American score...,54ebef61-7768-32ee-b9d7-35b4769957e1,[],NaN,[],,[],[],[],Voyage to the Bottom of the Sea,NaN,NaN,[],NaN,0,NaN
3,"[{'type': 'composer', 'begin': None, 'end': No...",67b6fe62-30ac-3052-b37e-45f38cc4ec55,[T-700.004.738-3],zxx,[],Star Trek: The Next Generation: The Best of Bo...,"[{'type': 'AGADU ID', 'type-id': '27ea7f59-9f9...",[zxx],[],Captain Borg,NaN,NaN,[],NaN,0,NaN
4,"[{'attribute-ids': {}, 'source-credit': '', 'd...",f80f0f84-0bfe-3d63-bd01-851b68e6c1b0,[],NaN,[],,[],[],[],A Whale of a Whale / Thar She Blows / A Whale ...,NaN,NaN,[],NaN,0,NaN


In [28]:
import json
import pandas as pd

records = []

with recording_file.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            records.append(json.loads(line))

        if len(records) >= 50000:
            break

recordings_df = pd.json_normalize(records)

print("Sample shape:", recordings_df.shape)
display(recordings_df.head())

Sample shape: (50000, 14)


,relations,id,artist-credit,disambiguation,tags,length,video,aliases,isrcs,title,annotation,genres,rating.value,rating.votes-count
0,"[{'target-credit': '', 'attribute-values': {},...",10b9c34b-821d-4cc8-9587-5c7d0cb68865,"[{'artist': {'sort-name': 'Pigface', 'genres':...",,[],NaN,False,[],[],Suck,NaN,[],NaN,0
1,"[{'target-credit': '', 'event': {'cancelled': ...",005495fc-81a2-4ce8-8f14-6ae442aec7f2,"[{'joinphrase': '', 'name': 'Phish', 'artist':...",,[],NaN,False,[],[],Oh Kee Pah Ceremony / Suzy Greenberg,NaN,[],NaN,0
2,[{'type-id': 'b06e6732-2603-47d3-8a49-9f589b43...,4a3ce0d6-5b8a-4d29-9320-6757922bed44,"[{'artist': {'type': 'Group', 'id': 'e01646f2-...",,[],NaN,False,[],[],Mike’s Song / I Am Hydrogen / Weekapaug Groove,NaN,[],NaN,0
3,[],565de6d8-f93f-4b44-97e2-c1f3ffc2410a,"[{'name': 'Baptism', 'joinphrase': '', 'artist...",,[],264000.0,False,[],[],The Calling,NaN,[],NaN,0
4,"[{'work': {'disambiguation': 'for orchestra', ...",33efcaf4-f7d5-41f6-b0d7-7771fa438821,[{'artist': {'type-id': 'ac897045-5043-3294-96...,,[],NaN,False,[],[],Finlandia,NaN,[],NaN,0


In [29]:
def names_from_list(value):
    """Get readable names from a list of strings or dictionaries."""
    if not isinstance(value, list):
        return []
    return [
        item if isinstance(item, str) else item.get("name")
        for item in value
        if isinstance(item, str) or (
            isinstance(item, dict) and item.get("name")
        )
    ]

# Keep useful work-level fields, if present in the dataframe.
work_columns = [
    "id", "title", "type", "language", "iswcs",
    "tags", "genres", "disambiguation", "rating.votes-count",
    "rating.value",
]
clean_works_df = works_df.reindex(columns=work_columns).copy()

clean_works_df = clean_works_df.rename(columns={
    "id": "work_mbid",
    "rating.votes-count": "rating_votes",
    "rating.value": "rating",
})

us_mbids = set(us_artists_df["artist_mbid"].dropna())

credit_rows = []
recording_work_rows = []

for _, row in recordings_df.iterrows():
    recording_mbid = row.get("id")

    # Collect US artists credited on this recording
    credits = row.get("artist-credit")
    if isinstance(credits, list):
        for credit in credits:
            if not isinstance(credit, dict):
                continue

            artist = credit.get("artist")
            if not isinstance(artist, dict):
                continue

            artist_mbid = artist.get("id") or artist.get("gid")
            if artist_mbid in us_mbids:
                credit_rows.append({
                    "recording_mbid": recording_mbid,
                    "artist_mbid": artist_mbid,
                    "artist_name": artist.get("name") or credit.get("name"),
                })

    # Find works linked to this recording
    relations = row.get("relations")
    if isinstance(relations, list):
        for relation in relations:
            if not isinstance(relation, dict):
                continue

            work = relation.get("work")
            if isinstance(work, dict):
                work_mbid = work.get("id") or work.get("gid")
                if work_mbid:
                    recording_work_rows.append({
                        "recording_mbid": recording_mbid,
                        "work_mbid": work_mbid,
                    })

recording_artist_credits_df = pd.DataFrame(credit_rows).drop_duplicates()
recording_work_links_df = pd.DataFrame(recording_work_rows).drop_duplicates()

# Associate US recording credits with the works linked to those recordings
work_recording_credits_df = recording_work_links_df.merge(
    recording_artist_credits_df,
    on="recording_mbid",
    how="inner",
).drop_duplicates()

# Keep only works that have at least one such US recording credit
us_credited_works_df = clean_works_df[
    clean_works_df["work_mbid"].isin(work_recording_credits_df["work_mbid"])
].copy()

print(f"Works with US recording credits: {len(us_credited_works_df)}")
print(f"US artist recording credits linked to works: {len(work_recording_credits_df)}")
display(us_credited_works_df.head())
display(work_recording_credits_df.head())

# # Convert nested values into readable text/list fields.
# for column in ["tags", "genres", "iswcs"]:
#     if column in clean_works_df.columns:
#         clean_works_df[column] = clean_works_df[column].apply(
#             lambda value: names_from_list(value) if isinstance(value, list) else []
#         )
#
# for column in ["title", "type", "language", "disambiguation"]:
#     if column in clean_works_df.columns:
#         clean_works_df[column] = clean_works_df[column].fillna("").astype(str).str.strip()
#
# clean_works_df["rating_votes"] = pd.to_numeric(
#     clean_works_df["rating_votes"], errors="coerce"
# ).fillna(0).astype(int)
#
# clean_works_df["rating"] = pd.to_numeric(
#     clean_works_df["rating"], errors="coerce"
# )
#
# # Unpack artist relations into one row per work–artist relationship.
# relation_rows = []
#
# for _, row in works_df.iterrows():
#     relations = row.get("relations")
#     if not isinstance(relations, list):
#         continue
#
#     for relation in relations:
#         if not isinstance(relation, dict):
#             continue
#
#         artist = relation.get("artist")
#         if not isinstance(artist, dict):
#             continue
#
#         relation_rows.append({
#             "work_mbid": row.get("id"),
#             "artist_mbid": artist.get("id") or artist.get("gid"),
#             "artist_name": artist.get("name"),
#             "relation_type": relation.get("type"),
#             "credited_as": relation.get("target-credit"),
#             "begin": relation.get("begin"),
#             "end": relation.get("end"),
#         })
#
# work_artist_relations_df = pd.DataFrame(
#     relation_rows,
#     columns=[
#         "work_mbid", "artist_mbid", "artist_name", "relation_type",
#         "credited_as", "begin", "end",
#     ],
# )
# us_mbids = set(us_artists_df["artist_mbid"].dropna())
#
# work_artist_relations_df = work_artist_relations_df[
#     work_artist_relations_df["artist_mbid"].isin(us_mbids)
# ].copy()
#
# # Keep only works that have at least one US artist relation
# clean_works_df = clean_works_df[
#     clean_works_df["work_mbid"].isin(work_artist_relations_df["work_mbid"])
# ].copy()
#
# print(f"US works: {len(clean_works_df)}")
# print(f"US work-artist relations: {len(work_artist_relations_df)}")
#
# display(clean_works_df.head())
# display(work_artist_relations_df.head())

Works with US recording credits: 2
US artist recording credits linked to works: 36


,work_mbid,title,type,language,iswcs,tags,genres,disambiguation,rating_votes,rating
19741,56dd2e8b-e5de-37cd-8b51-79ba0bc23a6e,雪に閉ざされて,NaN,zxx,[T-703.275.556-1],[],[],Buried in Snow - Final Fantasy VII,0,NaN
27590,894ae361-3d2b-3be3-bd27-dab78e107401,inner universe,Song,mul,[T-917.384.033-0],[],[],Ghost in the Shell: Stand Alone Complex,0,NaN


,recording_mbid,work_mbid,artist_mbid,artist_name
0,348d9121-7926-4418-b1da-65489ee86b80,31ad50c9-bcb5-440c-82c2-41d8203123c4,30e0b167-f506-40c2-ad5d-7bbea5607cdd,Joey Cape’s Bad Loud
1,e2091026-fa28-4026-8b6f-73731cbf5764,bbc9ce47-5d26-3489-ae0b-b06c4b8c6450,8aac35ef-e54d-4bcd-9682-e0cced4a6ea2,Work Drugs
2,040efa34-5d83-4be2-b361-70dcf29c00d2,1729345f-fb49-4fbb-b4cf-117d5098d381,98b45483-9735-4a31-9c19-6c551c4871eb,Singers Alumni
3,a4de3907-248b-4e86-93a0-b5c5a7b2b3f0,bc00e897-f38f-459b-88cb-cb2a16c99b24,98b45483-9735-4a31-9c19-6c551c4871eb,Singers Alumni
4,57fdd17a-bd82-40e3-a3eb-9bce6dbee43b,949ef2d8-ed22-44bc-8380-44fbe12f6e7c,98b45483-9735-4a31-9c19-6c551c4871eb,Singers Alumni


In [30]:
us_artist_lookup = (
    us_artists_df.dropna(subset=["artist_mbid"])
    .drop_duplicates("artist_mbid")
    .set_index("artist_mbid")
    .to_dict("index")
)

def list_values(value):
    """Normalize lists of strings or dictionaries into readable values."""
    if not isinstance(value, list):
        return []
    return [
        item if isinstance(item, str) else item.get("name")
        for item in value
        if isinstance(item, str) or (
            isinstance(item, dict) and item.get("name")
        )
    ]

recording_artist_rows = []

for _, recording in recordings_df.iterrows():
    recording_mbid = recording.get("id")

    # Collect works linked to this recording.
    works = []
    relations = recording.get("relations")
    if isinstance(relations, list):
        for relation in relations:
            if isinstance(relation, dict) and isinstance(relation.get("work"), dict):
                work = relation["work"]
                work_mbid = work.get("id") or work.get("gid")
                if work_mbid:
                    works.append({
                        "work_mbid": work_mbid,
                        "work_title": work.get("title"),
                    })

    # Create one row for each credited US artist.
    credits = recording.get("artist-credit")
    if not isinstance(credits, list):
        continue

    for credit in credits:
        if not isinstance(credit, dict):
            continue

        artist = credit.get("artist")
        if not isinstance(artist, dict):
            continue

        artist_mbid = artist.get("id") or artist.get("gid")
        if artist_mbid not in us_artist_lookup:
            continue

        artist_info = us_artist_lookup[artist_mbid]

        recording_artist_rows.append({
            "recording_mbid": recording_mbid,
            "recording_title": recording.get("title"),
            "artist_mbid": artist_mbid,
            "artist_name": artist_info.get("artist_name") or artist.get("name"),
            "artist_type": artist_info.get("artist_type"),
            "artist_country": artist_info.get("country"),
            "artist_begin_date": artist_info.get("begin_date"),
            "credited_as": credit.get("name"),
            "length_ms": recording.get("length"),
            "is_video": recording.get("video"),
            "isrcs": list_values(recording.get("isrcs")),
            "genres": list_values(recording.get("genres")),
            "tags": list_values(recording.get("tags")),
            "work_mbids": [work["work_mbid"] for work in works],
            "work_titles": [work["work_title"] for work in works if work.get("work_title")],
        })

recording_artist_df = pd.DataFrame(recording_artist_rows)

if not recording_artist_df.empty:
    recording_artist_df["length_minutes"] = (
        pd.to_numeric(recording_artist_df["length_ms"], errors="coerce") / 60_000
    )

print(f"Recording–US artist rows: {len(recording_artist_df)}")
pd.set_option("display.max_columns", None)
with pd.option_context("display.max_rows", 100, "display.max_columns", None):
    display(recording_artist_df.head(100))

Recording–US artist rows: 192


,recording_mbid,recording_title,artist_mbid,artist_name,artist_type,artist_country,artist_begin_date,credited_as,length_ms,is_video,isrcs,genres,tags,work_mbids,work_titles,length_minutes
0,348d9121-7926-4418-b1da-65489ee86b80,Okay,30e0b167-f506-40c2-ad5d-7bbea5607cdd,Joey Cape’s Bad Loud,Group,US,2011,Joey Cape’s Bad Loud,178000.0,False,[],[],[],[31ad50c9-bcb5-440c-82c2-41d8203123c4],[Okay],2.966667
1,333d1b8d-3714-4a07-b895-9e17555fea32,This Is Their Night,30e0b167-f506-40c2-ad5d-7bbea5607cdd,Joey Cape’s Bad Loud,Group,US,2011,Joey Cape’s Bad Loud,NaN,False,[],[],[],[],[],NaN
2,8d672183-8614-4159-babe-ab0c46cdc86b,Prince Blueblood,450c66a3-b431-44ae-8326-7e7f7db15bcd,The Beatle Bronies,Group,US,2011-08,The Beatle Bronies,53000.0,False,[],[],[],[],[],0.883333
3,b76e5226-b698-4a2c-a24a-564b0dfe53bf,The Quarantine Patrol,1f718a1d-cac8-4cc2-b9f2-63757d38dda4,Seeing Things,Group,US,2012,Seeing Things,180000.0,False,[],[],[],[],[],3.000000
4,67a269d5-bdd2-4c46-82f6-cb3cc3ab63bc,Doused,b5ff61dd-0154-4e80-b46b-c733f87db602,DIIV,Group,US,2011,DIIV,223000.0,False,[],[],[],[],[],3.716667
5,10d9af0f-41de-45e6-b1d3-733a7ed61285,How Long Have You Known?,b5ff61dd-0154-4e80-b46b-c733f87db602,DIIV,Group,US,2011,Dive,212000.0,False,[],[],[],[],[],3.533333
6,5782f0ea-5713-4bc8-b232-7beaaaa70fad,Sometime (Heavenly Beat edit),b5ff61dd-0154-4e80-b46b-c733f87db602,DIIV,Group,US,2011,Dive,201000.0,False,[],[],[],[],[],3.350000
7,dda96611-0899-4497-82df-8887470a105d,In the Hands of Godzilla (demo),14a2514d-0edd-4762-a649-d405baeeaa65,How The West Was Won,Group,US,2011,How The West Was Won,216000.0,False,[],[],[],[],[],3.600000
8,a72b2a44-8acb-4fe1-8e4a-373d90bbbc48,Day of the Ramrider,14a2514d-0edd-4762-a649-d405baeeaa65,How The West Was Won,Group,US,2011,How The West Was Won,191000.0,False,[],[],[],[],[],3.183333
9,7c141baa-ec4e-4b70-9513-a5422d9d5937,The Heron and the Fox (Work Drugs smooth mix),8aac35ef-e54d-4bcd-9682-e0cced4a6ea2,Work Drugs,Group,US,2011,Work Drugs,224000.0,False,[],[],[],[],[],3.733333
